# FASHN VTON 1.5 FastAPI server

This notebook exposes the already-installed FASHN VTON 1.5 pipeline through a small authenticated FastAPI service. Run the cells in order in the same Colab runtime where the pipeline and, if needed, `weights_dir` are already available.

The notebook does not clone the model repository, install `fashn-vton`, download model weights, or initialize a second pipeline when a usable `pipeline` object already exists. The tunnel setup is intentionally a separate phase; this notebook first verifies the service locally at port 8000.

## 1. Verify the GPU and reuse the loaded pipeline

The model is initialized only when `pipeline` is not already defined. If initialization is needed, this cell uses the existing `weights_dir` variable and does not download weights.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is not available. Select a GPU runtime in Colab and rerun this cell.")

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

existing_pipeline = globals().get("pipeline")
if existing_pipeline is not None and callable(existing_pipeline):
    pipeline = existing_pipeline
    print("Reusing the existing pipeline object.")
else:
    if "weights_dir" not in globals():
        raise RuntimeError(
            "No pipeline or weights_dir was found. Run your existing FASHN setup first; "
            "this notebook will not download model weights."
        )
    from fashn_vton import TryOnPipeline

    pipeline = TryOnPipeline(weights_dir=weights_dir, device="cuda")
    print("Initialized the pipeline from the existing weights_dir.")


## 2. Install only missing API dependencies

This checks for FastAPI, Uvicorn, and the multipart parser and installs only whichever of those are missing. It does not install or update the FASHN model package.

In [ ]:
import importlib.util
import subprocess
import sys

required_api_packages = {
    "fastapi": "fastapi",
    "uvicorn": "uvicorn",
    "multipart": "python-multipart",
}
missing_api_packages = [
    package_name
    for import_name, package_name in required_api_packages.items()
    if importlib.util.find_spec(import_name) is None
]

if missing_api_packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_api_packages])
else:
    print("FastAPI, Uvicorn, and python-multipart are already installed.")


## 3. Enter a shared server-only token

Generate a random token in a private terminal, for example with `python -c "import secrets; print(secrets.token_urlsafe(32))"`. Paste the same value into this password prompt and the Next.js server environment variable `COLAB_VTON_API_TOKEN`. The token is not printed or saved in this notebook. Do not use a `NEXT_PUBLIC_` variable.

In [ ]:
import getpass
import os

COLAB_VTON_API_TOKEN = os.environ.get("COLAB_VTON_API_TOKEN")
if not COLAB_VTON_API_TOKEN:
    COLAB_VTON_API_TOKEN = getpass.getpass("Paste the shared API token (input is hidden): ")

if len(COLAB_VTON_API_TOKEN) < 32:
    raise ValueError("Use a randomly generated token with at least 32 characters.")

os.environ["COLAB_VTON_API_TOKEN"] = COLAB_VTON_API_TOKEN
print("Shared token configured without displaying it.")


## 4. Define the authenticated API

Both endpoints require `Authorization: Bearer <token>`. Uploads are limited to 10 MiB per image and 20 MiB plus multipart overhead per request. Image contents are decoded and converted to RGB with Pillow. A non-blocking lock rejects overlapping inference requests instead of allowing them to compete for GPU memory. Internal exceptions are logged in Colab and are not returned to callers.

In [ ]:
import asyncio
import hmac
import io
import logging
import threading
import warnings

import torch
from fastapi import Depends, FastAPI, File, Form, Header, HTTPException, UploadFile
from fastapi.responses import JSONResponse, Response
from PIL import Image, UnidentifiedImageError

MAX_FILE_SIZE = 10 * 1024 * 1024
MAX_REQUEST_SIZE = 2 * MAX_FILE_SIZE + 256 * 1024
MAX_RESULT_SIZE = 15 * 1024 * 1024
MAX_IMAGE_PIXELS = 40_000_000
SUPPORTED_IMAGE_TYPES = {
    "image/jpeg": "JPEG",
    "image/png": "PNG",
    "image/webp": "WEBP",
}
SUPPORTED_CATEGORIES = {"tops", "bottoms", "one-pieces"}
SUPPORTED_GARMENT_PHOTO_TYPES = {"flat-lay", "model"}

Image.MAX_IMAGE_PIXELS = MAX_IMAGE_PIXELS
warnings.simplefilter("error", Image.DecompressionBombWarning)
logger = logging.getLogger("fashn_vton_api")
_inference_lock = threading.Lock()
_api_token = os.environ.get("COLAB_VTON_API_TOKEN")

if not _api_token or len(_api_token) < 32:
    raise RuntimeError("Run the token configuration cell before creating the API.")
if not callable(globals().get("pipeline")):
    raise RuntimeError("Run the GPU and pipeline setup cell before creating the API.")


async def require_api_token(authorization: str | None = Header(default=None)) -> None:
    scheme, separator, supplied_token = (authorization or "").partition(" ")
    if (
        not separator
        or scheme.lower() != "bearer"
        or not hmac.compare_digest(supplied_token, _api_token)
    ):
        raise HTTPException(
            status_code=401,
            detail="A valid bearer token is required.",
            headers={"WWW-Authenticate": "Bearer"},
        )


class RequestBodyTooLarge(Exception):
    pass


class RequestSizeLimitMiddleware:
    def __init__(self, app, max_body_size: int):
        self.app = app
        self.max_body_size = max_body_size

    async def __call__(self, scope, receive, send):
        if not (
            scope["type"] == "http"
            and scope["path"] == "/virtual-try-on"
            and scope["method"] == "POST"
        ):
            await self.app(scope, receive, send)
            return

        headers = dict(scope.get("headers", []))
        content_length = headers.get(b"content-length")
        if content_length is not None:
            try:
                if int(content_length) > self.max_body_size:
                    response = JSONResponse(
                        status_code=413,
                        content={"detail": "The combined upload is too large."},
                    )
                    await response(scope, receive, send)
                    return
            except ValueError:
                response = JSONResponse(
                    status_code=400,
                    content={"detail": "Invalid Content-Length header."},
                )
                await response(scope, receive, send)
                return

        received_size = 0

        async def receive_with_limit():
            nonlocal received_size
            message = await receive()
            if message["type"] == "http.request":
                received_size += len(message.get("body", b""))
                if received_size > self.max_body_size:
                    raise RequestBodyTooLarge
            return message

        try:
            await self.app(scope, receive_with_limit, send)
        except RequestBodyTooLarge:
            response = JSONResponse(
                status_code=413,
                content={"detail": "The combined upload is too large."},
            )
            await response(scope, receive, send)


app = FastAPI(docs_url=None, redoc_url=None, openapi_url=None)
app.add_middleware(RequestSizeLimitMiddleware, max_body_size=MAX_REQUEST_SIZE)


async def read_limited_upload(upload: UploadFile) -> bytes:
    chunks = []
    total_size = 0
    while True:
        chunk = await upload.read(min(1024 * 1024, MAX_FILE_SIZE + 1 - total_size))
        if not chunk:
            break
        total_size += len(chunk)
        if total_size > MAX_FILE_SIZE:
            raise HTTPException(status_code=413, detail="Each image must be 10 MiB or smaller.")
        chunks.append(chunk)

    if total_size == 0:
        raise HTTPException(status_code=400, detail="Both image files must be non-empty.")
    return b"".join(chunks)


async def decode_rgb_image(upload: UploadFile) -> Image.Image:
    expected_format = SUPPORTED_IMAGE_TYPES.get(upload.content_type or "")
    if expected_format is None:
        raise HTTPException(status_code=415, detail="Images must be JPEG, PNG, or WebP.")

    image_bytes = await read_limited_upload(upload)
    try:
        with Image.open(io.BytesIO(image_bytes)) as image:
            if image.format != expected_format:
                raise HTTPException(status_code=415, detail="The image content does not match its file type.")
            image.verify()
        with Image.open(io.BytesIO(image_bytes)) as image:
            if image.width * image.height > MAX_IMAGE_PIXELS:
                raise HTTPException(status_code=413, detail="The image dimensions are too large.")
            image.load()
            return image.convert("RGB").copy()
    except HTTPException:
        raise
    except (
        Image.DecompressionBombError,
        Image.DecompressionBombWarning,
        UnidentifiedImageError,
        OSError,
        ValueError,
    ) as exc:
        raise HTTPException(status_code=415, detail="The uploaded image is invalid or too large to decode.") from exc


class InferenceBusyError(Exception):
    pass


def run_pipeline(person: Image.Image, garment: Image.Image, category: str, photo_type: str):
    if not _inference_lock.acquire(blocking=False):
        raise InferenceBusyError
    try:
        result = pipeline(
            person_image=person,
            garment_image=garment,
            category=category,
            garment_photo_type=photo_type,
            num_samples=1,
            num_timesteps=30,
            seed=42,
        )
        result_images = getattr(result, "images", None)
        if not result_images or not isinstance(result_images[0], Image.Image):
            raise RuntimeError("The pipeline did not return a PIL image.")
        return result_images[0].copy()
    finally:
        _inference_lock.release()


@app.get("/health", dependencies=[Depends(require_api_token)])
async def health():
    return {
        "status": "ok",
        "gpu_available": torch.cuda.is_available(),
        "pipeline_loaded": callable(globals().get("pipeline")),
    }


@app.post("/virtual-try-on", dependencies=[Depends(require_api_token)])
async def virtual_try_on(
    person_image: UploadFile = File(...),
    garment_image: UploadFile = File(...),
    category: str = Form(...),
    garment_photo_type: str = Form(...),
):
    if category not in SUPPORTED_CATEGORIES:
        raise HTTPException(status_code=422, detail="Category must be tops, bottoms, or one-pieces.")
    if garment_photo_type not in SUPPORTED_GARMENT_PHOTO_TYPES:
        raise HTTPException(status_code=422, detail="Garment photo type must be flat-lay or model.")

    person = await decode_rgb_image(person_image)
    garment = await decode_rgb_image(garment_image)

    try:
        output_image = await asyncio.to_thread(run_pipeline, person, garment, category, garment_photo_type)
    except InferenceBusyError as exc:
        raise HTTPException(status_code=503, detail="Another fitting is running. Please retry shortly.") from exc
    except torch.cuda.OutOfMemoryError as exc:
        logger.exception("FASHN VTON ran out of GPU memory.")
        torch.cuda.empty_cache()
        raise HTTPException(status_code=503, detail="The GPU is temporarily unavailable. Please retry shortly.") from exc
    except Exception as exc:
        logger.exception("FASHN VTON inference failed.")
        raise HTTPException(status_code=502, detail="The fitting could not be completed. Please try again.") from exc

    try:
        output_buffer = io.BytesIO()
        output_image.convert("RGB").save(output_buffer, format="PNG", optimize=True)
        output_bytes = output_buffer.getvalue()
    except Exception as exc:
        logger.exception("Could not encode the FASHN result as PNG.")
        raise HTTPException(status_code=502, detail="The fitting result could not be encoded.") from exc

    if not output_bytes:
        raise HTTPException(status_code=502, detail="The fitting returned an empty image.")
    if len(output_bytes) > MAX_RESULT_SIZE:
        raise HTTPException(status_code=502, detail="The generated image exceeds the response size limit.")

    return Response(
        content=output_bytes,
        media_type="image/png",
        headers={"Cache-Control": "no-store", "X-Content-Type-Options": "nosniff"},
    )


## 5. Start Uvicorn on port 8000

The server runs in a background notebook thread so later cells remain usable. It binds to `0.0.0.0:8000` as required, uses one Uvicorn process, and therefore shares the single-inference lock.

In [ ]:
import threading
import time
import uvicorn

if globals().get("uvicorn_thread") is not None and uvicorn_thread.is_alive():
    print("The FastAPI server is already running on port 8000.")
else:
    uvicorn_config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info", workers=1)
    uvicorn_server = uvicorn.Server(uvicorn_config)
    uvicorn_thread = threading.Thread(target=uvicorn_server.run, daemon=True)
    uvicorn_thread.start()

    for _ in range(100):
        if uvicorn_server.started:
            break
        if not uvicorn_thread.is_alive():
            raise RuntimeError("Uvicorn exited before the server became ready. Check the cell output.")
        time.sleep(0.1)

    if not uvicorn_server.started:
        raise TimeoutError("Uvicorn did not become ready within 10 seconds.")
    print("FastAPI is listening on http://0.0.0.0:8000")


## 6. Test the authenticated health endpoint locally

A successful response confirms the local API can see both the GPU and the existing pipeline. The unauthenticated check should return HTTP 401.

In [ ]:
from urllib.error import HTTPError
from urllib.request import Request, urlopen

health_request = Request(
    "http://127.0.0.1:8000/health",
    headers={"Authorization": f"Bearer {COLAB_VTON_API_TOKEN}"},
)
with urlopen(health_request, timeout=10) as response:
    print("Authenticated health status:", response.status)
    print(response.read().decode("utf-8"))

try:
    with urlopen("http://127.0.0.1:8000/health", timeout=10) as response:
        raise AssertionError(f"Expected unauthenticated request to fail, got HTTP {response.status}.")
except HTTPError as exc:
    if exc.code != 401:
        raise
    print("Unauthenticated health check correctly returned HTTP 401.")


## 7. Test a real inference locally

Upload one person photo and one garment photo. This cell sends both files to the local endpoint, checks that the result is a bounded PNG, saves it under `/content/vton_result.png`, and displays it. Set the category and garment photo type to match the selected garment. The provided model example uses `tops` and `flat-lay`.

In [ ]:
from google.colab import files
from IPython.display import display
from PIL import Image
from urllib.request import Request, urlopen
import json

TEST_CATEGORY = "tops"  # tops, bottoms, or one-pieces
TEST_GARMENT_PHOTO_TYPE = "flat-lay"  # flat-lay or model

uploaded_files = files.upload()
if len(uploaded_files) != 2:
    raise ValueError("Upload exactly two files: a person photo and a garment photo.")

uploaded_items = list(uploaded_files.items())
person_name, person_bytes = uploaded_items[0]
garment_name, garment_bytes = uploaded_items[1]

def image_content_type(image_bytes):
    with Image.open(io.BytesIO(image_bytes)) as image:
        content_type = {
            "JPEG": "image/jpeg",
            "PNG": "image/png",
            "WEBP": "image/webp",
        }.get(image.format)
    if content_type is None:
        raise ValueError("Test images must be JPEG, PNG, or WebP.")
    return content_type

boundary = "----ColabVTONBoundary7MA4YWxkTrZu0gW"
multipart_parts = []

def add_form_field(name, value):
    multipart_parts.extend([
        f"--{boundary}\r\n".encode(),
        f'Content-Disposition: form-data; name="{name}"\r\n\r\n'.encode(),
        str(value).encode(),
        b"\r\n",
    ])

def add_form_file(field_name, filename, image_bytes):
    content_type = image_content_type(image_bytes)
    multipart_parts.extend([
        f"--{boundary}\r\n".encode(),
        f'Content-Disposition: form-data; name="{field_name}"; filename="{filename}"\r\n'.encode(),
        f"Content-Type: {content_type}\r\n\r\n".encode(),
        image_bytes,
        b"\r\n",
    ])

add_form_field("category", TEST_CATEGORY)
add_form_field("garment_photo_type", TEST_GARMENT_PHOTO_TYPE)
add_form_file("person_image", "person-image", person_bytes)
add_form_file("garment_image", "garment-image", garment_bytes)
multipart_parts.append(f"--{boundary}--\r\n".encode())

tryon_request = Request(
    "http://127.0.0.1:8000/virtual-try-on",
    data=b"".join(multipart_parts),
    headers={
        "Authorization": f"Bearer {COLAB_VTON_API_TOKEN}",
        "Content-Type": f"multipart/form-data; boundary={boundary}",
    },
    method="POST",
)

with urlopen(tryon_request, timeout=300) as response:
    if response.status != 200 or response.headers.get_content_type() != "image/png":
        raise RuntimeError(f"Unexpected model response: HTTP {response.status}, {response.headers.get_content_type()}.")
    result_bytes = response.read(MAX_RESULT_SIZE + 1)

if not result_bytes or len(result_bytes) > MAX_RESULT_SIZE:
    raise ValueError("The endpoint returned an empty or oversized result.")
if not result_bytes.startswith(b"\x89PNG\r\n\x1a\n"):
    raise ValueError("The endpoint response does not contain a valid PNG signature.")

result_path = "/content/vton_result.png"
with open(result_path, "wb") as result_file:
    result_file.write(result_bytes)
print(f"Saved {len(result_bytes):,} bytes to {result_path}")
display(Image.open(io.BytesIO(result_bytes)))


## 8. Install Cloudflare Tunnel for development

Cloudflare [Quick Tunnels](https://developers.cloudflare.com/tunnel/get-started/quick-tunnels/) create a temporary public `trycloudflare.com` hostname without an account or a named domain. Anyone can reach that hostname, so both API routes still require the bearer token. This is for development only, has no uptime guarantee, and the hostname changes when the tunnel or Colab runtime restarts. The tunnel forwards HTTPS traffic to the local `http://127.0.0.1:8000` server.

The next cell downloads the official [Cloudflare Linux x86-64 binary](https://developers.cloudflare.com/tunnel/downloads/) only if `cloudflared` is not already installed.

In [ ]:
import os
import platform
import shutil
import subprocess
from urllib.request import urlretrieve

if platform.machine().lower() not in {"x86_64", "amd64"}:
    raise RuntimeError(f"This install cell expects a Linux x86-64 Colab runtime, got {platform.machine()}.")

cloudflared_binary = shutil.which("cloudflared")
if cloudflared_binary is None:
    cloudflared_binary = "/content/cloudflared"
    urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        cloudflared_binary,
    )
    os.chmod(cloudflared_binary, 0o755)

cloudflared_version = subprocess.run(
    [cloudflared_binary, "--version"], check=True, capture_output=True, text=True
)
print(cloudflared_version.stdout.strip() or cloudflared_version.stderr.strip())


## 9. Start the tunnel and capture its public URL

Run this only after the local FastAPI health test passes. Copy the printed URL into the Next.js server-side `COLAB_VTON_URL` setting (for local development, `.env.local`). Keep it out of browser code and never name it `NEXT_PUBLIC_COLAB_VTON_URL`. The token is not printed.

In [ ]:
import collections
import queue
import re
import threading
import time

if (
    globals().get("cloudflared_process") is not None
    and cloudflared_process.poll() is None
    and os.environ.get("COLAB_VTON_URL")
):
    COLAB_VTON_URL = os.environ["COLAB_VTON_URL"]
    print(f"Existing tunnel is running: {COLAB_VTON_URL}")
else:
    tunnel_url_queue = queue.Queue(maxsize=1)
    tunnel_log_tail = collections.deque(maxlen=40)
    cloudflared_process = subprocess.Popen(
        [cloudflared_binary, "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8000"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    def read_cloudflared_output():
        for line in cloudflared_process.stdout:
            tunnel_log_tail.append(line.rstrip())
            match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
            if match:
                try:
                    tunnel_url_queue.put_nowait(match.group(0))
                except queue.Full:
                    pass

    cloudflared_reader = threading.Thread(target=read_cloudflared_output, daemon=True)
    cloudflared_reader.start()

    try:
        COLAB_VTON_URL = tunnel_url_queue.get(timeout=120).rstrip("/")
    except queue.Empty as exc:
        cloudflared_process.terminate()
        raise TimeoutError(
            "cloudflared did not print a Quick Tunnel URL. Recent output:\n"
            + "\n".join(tunnel_log_tail)
        ) from exc

    os.environ["COLAB_VTON_URL"] = COLAB_VTON_URL
    print(f"Public tunnel URL: {COLAB_VTON_URL}")
    print("Set this as server-only COLAB_VTON_URL in Next.js; the URL can change after a runtime restart.")


## 10. Verify the public tunnel and authentication

This tests the authenticated health endpoint through Cloudflare and confirms that the public route rejects a request without the shared token.

In [ ]:
import json
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

public_health_request = Request(
    f"{COLAB_VTON_URL}/health",
    headers={"Authorization": f"Bearer {COLAB_VTON_API_TOKEN}"},
)

for attempt in range(30):
    try:
        with urlopen(public_health_request, timeout=20) as response:
            public_health = json.loads(response.read().decode("utf-8"))
            if response.status != 200 or public_health.get("status") != "ok":
                raise RuntimeError("The public tunnel health response was unexpected.")
            print("Authenticated public health check:", public_health)
            break
    except (URLError, TimeoutError):
        if attempt == 29:
            raise
        time.sleep(2)

try:
    with urlopen(f"{COLAB_VTON_URL}/health", timeout=20) as response:
        raise AssertionError(f"Expected HTTP 401 without a token, got HTTP {response.status}.")
except HTTPError as exc:
    if exc.code != 401:
        raise
    print("Unauthenticated public health check correctly returned HTTP 401.")


## 11. Verify inference through the public tunnel

Run after the local inference test cell has uploaded the person and garment samples. This sends the same images to the public tunnel and confirms that the returned response is a bounded PNG.

In [ ]:
if "person_bytes" not in globals() or "garment_bytes" not in globals():
    raise RuntimeError("Run the local inference test cell first so the two sample images are available.")

public_boundary = "----ColabVTONPublicBoundary7MA4YWxkTrZu0gW"
public_parts = []

def add_public_field(name, value):
    public_parts.extend([
        f"--{public_boundary}\r\n".encode(),
        f'Content-Disposition: form-data; name="{name}"\r\n\r\n'.encode(),
        str(value).encode(),
        b"\r\n",
    ])

def add_public_file(field_name, filename, image_bytes):
    public_parts.extend([
        f"--{public_boundary}\r\n".encode(),
        f'Content-Disposition: form-data; name="{field_name}"; filename="{filename}"\r\n'.encode(),
        f"Content-Type: {image_content_type(image_bytes)}\r\n\r\n".encode(),
        image_bytes,
        b"\r\n",
    ])

add_public_field("category", TEST_CATEGORY)
add_public_field("garment_photo_type", TEST_GARMENT_PHOTO_TYPE)
add_public_file("person_image", "person-image", person_bytes)
add_public_file("garment_image", "garment-image", garment_bytes)
public_parts.append(f"--{public_boundary}--\r\n".encode())

public_tryon_request = Request(
    f"{COLAB_VTON_URL}/virtual-try-on",
    data=b"".join(public_parts),
    headers={
        "Authorization": f"Bearer {COLAB_VTON_API_TOKEN}",
        "Content-Type": f"multipart/form-data; boundary={public_boundary}",
    },
    method="POST",
)
with urlopen(public_tryon_request, timeout=300) as response:
    if response.status != 200 or response.headers.get_content_type() != "image/png":
        raise RuntimeError("The public inference endpoint returned an unexpected response.")
    public_result = response.read(MAX_RESULT_SIZE + 1)

if not public_result or len(public_result) > MAX_RESULT_SIZE:
    raise ValueError("The public endpoint returned an empty or oversized result.")
if not public_result.startswith(b"\x89PNG\r\n\x1a\n"):
    raise ValueError("The public endpoint response does not contain a valid PNG signature.")

public_result_path = "/content/vton_result_tunnel.png"
with open(public_result_path, "wb") as result_file:
    result_file.write(public_result)
print(f"Public tunnel inference succeeded: {len(public_result):,} bytes saved to {public_result_path}")
display(Image.open(io.BytesIO(public_result)))


## Stop the tunnel when finished

Stopping the tunnel makes its public URL unavailable. A later Quick Tunnel will use a different URL; update the server-side `COLAB_VTON_URL` value in Next.js accordingly.

In [ ]:
if globals().get("cloudflared_process") is not None and cloudflared_process.poll() is None:
    cloudflared_process.terminate()
    try:
        cloudflared_process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        cloudflared_process.kill()
        cloudflared_process.wait(timeout=5)
    print("Cloudflare Tunnel stopped.")
else:
    print("No Cloudflare Tunnel is running.")


## Stop the FastAPI server when finished

Run this cell when you are finished with the local test server. Colab will also stop the server when its runtime ends.

In [ ]:
if globals().get("uvicorn_server") is not None:
    uvicorn_server.should_exit = True
    if globals().get("uvicorn_thread") is not None:
        uvicorn_thread.join(timeout=10)
    print("Uvicorn shutdown requested.")
else:
    print("No Uvicorn server is running.")
